In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# Stage B: construct paper-matched variables for USA
# Input  : DATA/processed/USA_stageA.parquet
# Output : DATA/processed/USA_stageB.parquet
# =========================================================

# -----------------------------
# Paths
# -----------------------------
base = Path.cwd()
in_path = base / "DATA" / "processed" / "USA_stageA.parquet"
out_dir = base / "DATA" / "processed"
out_path = out_dir / "USA_stageB.parquet"
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load Stage A
# -----------------------------
df = pd.read_parquet(in_path).copy()
df["eom"] = pd.to_datetime(df["eom"], errors="coerce")

# make sure panel is sorted before lagging / shifting
df = df.sort_values(["permno", "eom"]).reset_index(drop=True)

# -----------------------------
# Helper functions
# -----------------------------
group_cols = ["eom", "ff49"]  # within month + industry for USA


def leave_one_out_mean(data: pd.DataFrame, var: str, by: list[str]) -> pd.Series:
    """
    Leave-one-out mean within groups.
    For firm i in group g:
        (sum_g - x_i) / (n_g - 1)
    Returns NaN if group has <= 1 non-missing observation.
    """
    x = pd.to_numeric(data[var], errors="coerce")
    grp = x.groupby([data[c] for c in by])

    group_sum = grp.transform("sum")
    group_count = grp.transform("count")

    loo = pd.Series(np.nan, index=data.index, dtype="float64")
    mask = x.notna() & (group_count > 1)
    loo.loc[mask] = (group_sum.loc[mask] - x.loc[mask]) / (group_count.loc[mask] - 1)
    return loo


def industry_herf_sales(data: pd.DataFrame, sales_col: str, by: list[str]) -> pd.Series:
    """
    Herfindahl concentration within month-industry:
        sum_j (sales_j / total_sales_group)^2
    Same value is assigned to all stocks in the same group.
    """
    sales = pd.to_numeric(data[sales_col], errors="coerce")
    grp = sales.groupby([data[c] for c in by])

    total_sales = grp.transform("sum")
    valid = sales.notna() & total_sales.gt(0)

    share = pd.Series(np.nan, index=data.index, dtype="float64")
    share.loc[valid] = sales.loc[valid] / total_sales.loc[valid]

    share_sq = share ** 2
    herf = share_sq.groupby([data[c] for c in by]).transform("sum")
    return herf


# -----------------------------
# Base variables needed for transformations
# -----------------------------

# mvel1 = log market equity
df["mvel1"] = np.where(df["market_equity"] > 0, np.log(df["market_equity"]), np.nan)

# 12-month change in profit margin per stock
# assumes one row per stock-month after Stage A
df["pm_change_12m"] = df.groupby("permno")["ebit_sale"].shift(0) - df.groupby("permno")["ebit_sale"].shift(12)

# leave-one-out industry means
df["be_me_ind_mean"] = leave_one_out_mean(df, "be_me", group_cols)
df["fcf_me_ind_mean"] = leave_one_out_mean(df, "fcf_me", group_cols)
df["mvel1_ind_mean"] = leave_one_out_mean(df, "mvel1", group_cols)
df["pm_change_12m_ind_mean"] = leave_one_out_mean(df, "pm_change_12m", group_cols)

# industry equal-weighted 12-month momentum (excluding self)
df["indmom_a_12"] = leave_one_out_mean(df, "ret_12_1", group_cols)

# industry sales concentration
df["herf"] = industry_herf_sales(df, "sales", group_cols)

# cash level for salecash
df["cash_level"] = df["cash_at"] * df["assets"]
df["salecash"] = np.where(df["cash_level"] > 0, df["sales"] / df["cash_level"], np.nan)

# -----------------------------
# Construct paper-matched variables
# -----------------------------
stageB = pd.DataFrame(index=df.index)

# identifiers
stageB["permno"] = df["permno"]
stageB["gvkey"] = df["gvkey"]
stageB["eom"] = df["eom"]
stageB["excntry"] = df["excntry"]
stageB["ff49"] = df["ff49"]

# paper variables
stageB["absacc"] = df["oaccruals_at"].abs()
stageB["acc"] = df["oaccruals_at"]
stageB["agr"] = df["at_gr1"]
stageB["bm"] = df["be_me"]
stageB["bm_ia"] = df["be_me"] - df["be_me_ind_mean"]
stageB["cashdebt"] = df["ocf_debt"]

# unresolved / omitted
stageB["cashpr"] = np.nan
stageB["depr"] = np.nan

stageB["cfp"] = df["fcf_me"]
stageB["cfp_ia"] = df["fcf_me"] - df["fcf_me_ind_mean"]
stageB["chmom_6"] = df["ret_6_1"] - df["ret_12_7"]

# FIXED: change in profit margin, then industry-adjust
stageB["chpmia"] = df["pm_change_12m"] - df["pm_change_12m_ind_mean"]

stageB["dolvol"] = df["dolvol_126d"]
stageB["dy"] = df["div12m_me"]
stageB["egr"] = df["be_gr1a"]
stageB["ep"] = df["ni_me"]
stageB["herf"] = df["herf"]
stageB["ill"] = df["ami_126d"]
stageB["indmom_a_12"] = df["indmom_a_12"]
stageB["lev"] = df["at_be"]
stageB["lgr"] = df["debtlt_gr1a"]
stageB["maxret"] = df["rmax1_21d"]
stageB["mom_1"] = df["ret_1_0"]
stageB["mom_12"] = df["ret_12_1"]
stageB["mom_6"] = df["ret_6_1"]
stageB["mve_ia"] = df["mvel1"] - df["mvel1_ind_mean"]
stageB["mvel1"] = df["mvel1"]
stageB["pctacc"] = df["taccruals_ni"]
stageB["retvol"] = df["rvol_21d"]
stageB["roe"] = df["ni_be"]
stageB["salecash"] = df["salecash"]
stageB["sgr"] = df["sale_gr1"]
stageB["sp"] = df["sale_me"]
stageB["stddolvol"] = df["dolvol_var_126d"]
stageB["stdturn"] = df["turnover_var_126d"]
stageB["turn"] = df["turnover_126d"]

# -----------------------------
# Quick QA
# -----------------------------
paper_vars = [
    "absacc", "acc", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
    "chmom_6", "chpmia", "depr", "dolvol", "dy", "egr", "ep", "herf", "ill",
    "indmom_a_12", "lev", "lgr", "maxret", "mom_1", "mom_12", "mom_6", "mve_ia",
    "mvel1", "pctacc", "retvol", "roe", "salecash", "sgr", "sp", "stddolvol",
    "stdturn", "turn"
]

print("\n=========================")
print("STAGE B SUMMARY")
print("=========================")
print("Shape:", stageB.shape)
print("Min eom:", stageB["eom"].min())
print("Max eom:", stageB["eom"].max())

print("\nMissing share by paper variable:")
print(stageB[paper_vars].isna().mean().sort_values(ascending=False))

print("\nDuplicate permno-eom rows:")
print(stageB.duplicated(["permno", "eom"]).sum())

print("\nQuick check of chpmia inputs:")
print(df[["ebit_sale", "pm_change_12m", "pm_change_12m_ind_mean"]].describe())

print("\nSample rows:")
print(stageB.head())

# -----------------------------
# Save
# -----------------------------
stageB.to_parquet(out_path, index=False)

print("\nSaved Stage B file to:")
print(out_path)


/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel_1932/2369839071.py:24: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["eom"] = pd.to_datetime(df["eom"], errors="coerce")
/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel


STAGE B SUMMARY
Shape: (3191806, 41)
Min eom: 1963-01-31 00:00:00
Max eom: 2024-12-31 00:00:00

Missing share by paper variable:
cashpr         1.000000
depr           1.000000
ill            0.130989
retvol         0.125248
maxret         0.125248
cashdebt       0.115924
chpmia         0.113598
indmom_a_12    0.092000
cfp_ia         0.091556
pctacc         0.091332
egr            0.082888
acc            0.082550
absacc         0.082550
stddolvol      0.076748
stdturn        0.076747
dolvol         0.076741
turn           0.076740
mom_12         0.066846
chmom_6        0.066634
cfp            0.065094
bm_ia          0.057947
dy             0.053616
sgr            0.048812
lgr            0.037832
agr            0.034673
roe            0.030988
lev            0.030601
bm             0.030598
mve_ia         0.028615
herf           0.028586
mom_6          0.028383
mom_1          0.005264
ep             0.000009
mvel1          0.000000
salecash       0.000000
sp             0.000000
dtype: